<a href="https://colab.research.google.com/github/kasettisiva/R_training_summer_workshop_2026/blob/main/notebooks/03_Data_Structures_and_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

#
# **LONI Scientific Computing Bootcamp 2026**

## **Introduction to R**

*A hands-on workshop for data analysis, visualization, and statistical computing.*

---

**Siva Prasad Kasetti**

**LSU/LONI HPC User Services**  
**skasetti1@lsu.edu**  
**08 June 2026**

</div>

---

# **Chapter 3: Data Structures & Data Wrangling**

This notebook covers the core R data structures (vectors, data frames, subsetting) and hands-on data wrangling with `dplyr`.

**Sections:** Vectors · Data Frames · Subsetting · Data Cleaning · Data Manipulation · dplyr verbs · Pipe operator

---

## **Run This First**

Install and load all packages needed for the workshop.

**DO NOT CHANGE THE RUNTIME BECAUSE YOU WON'T BE ABLE TO CHANGE IT BACK!**

In [ ]:
#@markdown Install packages
install.packages(c("ggplot2", "dplyr", "tidyr", "rpart", "rpart.plot", "Metrics"))

library(ggplot2)
library(dplyr)
library(tidyr)
library(rpart)
library(rpart.plot)
library(Metrics)
print("All packages loaded.\n")

---

# **3 · Data Structures That You'll Actually Use**



Before we even look into R data structures, let's look at what are the atomic data types.

### **3.1 Atomic Data Types**

R has five atomic data classes:



* Numeric (double)
  * Numbers in R are treated as numeric unless specified otherwise.

In [ ]:
# The class() function reveals the class of a R object.
class(9.3)
class(3)

* Integer

In [ ]:
# The as.integer() function "casts" an object into integer.
class(as.integer(3))

* Complex

In [ ]:
class(3+2i)

* Character

In [ ]:
# Both are categorized as characters.
class("a")
class("a cat")

* Logical (T, TRUE, F, FALSE)
  * Note that they must be upper case



In [ ]:
class(TRUE)
class(T)
class(True)

`NA` is a logical constant R uses to denote a missing value.

In [ ]:
class(NA)

The `is.<type>()` functions, which return logical values, can be used to check for the data classes too.

In [ ]:
a <- 3
# Is a numeric?
is.numeric(a)
# Is a integer?
is.integer(a)
# Is a logical?
is.logical(a)

Now let's look at the data objects in R. They are:

* Vectors: elements of same class, one dimension
* Matrices: elements of same class, two dimensions
* Arrays: elements of same class, 2+ dimensions
* Lists: elements can be any objects
* Data frames: “datasets” where columns are variables and rows are observations

> We focus on the two you'll encounter 95% of the time:
**vectors** and **data frames**.

### **3.2 Vectors**
A vector is the most fundamental data structure in R. Vectors are one-dimentional arrays that contain elements of the **same** data type. Almost everything in R is built on vectors.

Vectors can be constructed by
* The `c()` funtion (concatenate):

In [ ]:
#@markdown **Numeric Vector**
# Numeric vector — e.g. expression values for 5 genes
expr <- c(12.4, 0.3, 88.1, 45.0, 7.7)
expr

# Character vector — gene names
genes <- c("BRCA1", "TP53", "MYC", "EGFR", "KRAS")
genes

In [ ]:
#@markdown **Logical Vector**
# Logical vector — which genes are highly expressed (> 40)?
high_expr <- expr > 40
high_expr

# Use that logical vector to filter
genes[high_expr]   # <- This pattern is everywhere in R

* The `seq()` and `rep()` functions, or the `:` operator

In [ ]:
#@markdown **seq()**
# seq() — evenly spaced sequences
seq(1, 10, by = 2)        # 1 3 5 7 9
seq(0, 1, length.out = 5) # 0.00 0.25 0.50 0.75 1.00

In [ ]:
#@markdown **rep()**
# rep() — repeat values
rep(0, times = 5)          # 0 0 0 0 0
rep(c(1, 2), times = 3)    # 1 2 1 2 1 2
rep(c(1, 2), each = 3)     # 1 1 1 2 2 2

In [ ]:
#@markdown **":" operator**
# : operator — integer range
1:6                        # 1 2 3 4 5 6

# combined all of them
c(1,8:10,rep(2:4,3))

In [ ]:
#@markdown **vector()**
# vector() — preallocate an empty vector
x <- vector("numeric", length = 5)   # 0 0 0 0 0
x
y <- vector("character", length = 3) # "" "" ""
y

You can convert an object to a different type using the `as.<TYPE>()` functions.

Note: the output will be an object of the specified type, while the input remains untouched.

In [ ]:
# myvec is an integer vector.
myvec <- 1:3
myvec

In [ ]:
# The output of as.character(myvec) is a character vector.
as.character(myvec)

When converting to logical values, a numeric/integer "0" will be FALSE while all non-zeroes will be TRUE.

In [ ]:
as.logical(0:3)

Coercion will occur when mixed objects are passed to the `c()` function, as if an `as.<Type>()` function is explicitly called.

In [ ]:
# Which data type would myvec be?
myvec <- c(1e3,"a")
myvec
#class(myvec)

In [ ]:
# How about this one?
c(1.7,"a")

In [ ]:
# How about this one?
c(T,2)

Coercion hierarchy in R:

logical < integer < numeric < character

Caution: type coercion may happen **without you being aware of it and may have unintended results**.

**Access Vector Elements**

One can use the `[<index>]` operator to access individual element in a vector. <font color=red>**Note**</font> that in R the indices start from 1.

In [ ]:
myvec <- 1:10
# myvec[4] points to the 4th element in the vector.
myvec[4]

For multiple elements with multiple indices, use the `c()` function (the indices need to be passed as a vector themselves).

In [ ]:
# This will not work:
myvec[1,4]

# But this will:
myvec[c(1,4)]

Negative indices will drop the corresponding elements from the vector.

In [ ]:
# Drop element 2 through 6.
myvec[-2:-6]

Logical values can be used to access individual elements too.

In [ ]:
#@markdown What output do you expect to see? **myvec[c(T,F)]**
#R recycles the logical vector to match the length of the vector
myvec[c(T,F)]

<font color=red>**Important:**</font> Lots of R operations process objects in a vectorized way
* More efficient, concise, and easier to read.

In [ ]:
# vec1: 1 2 3 4
# vec2: 4 3 2 1
vec1 <- 1:4
vec2 <- 4:1

In [ ]:
# Element-wise addition
vec1 + vec2

In [ ]:
# Element-wise comparison
# The result will be a logical vector with four elements.
vec1 > 2

In [ ]:
# Element-wise comparison
vec1 > vec2

In [ ]:
# Equivalent to "show all vec1 elements that are greater than their counterparts in vec2".
vec1[vec1 > vec2]

Sometimes R takes it to the extreme. It's called "**recycling**".

In [ ]:
# vec1: 1 2 3 4 5 6 7 8
# vec2: 4 3 2 1
vec1 <- 1:8
vec2 <- 4:1
# Now vec1 and vec2 are of different length. Would this end up with an error?
vec1+vec2

# The shorter vector (vec2) will be recycled.
# R will add (4,3,2,1,4,3,2,1) to (1,2,3,4,5,6,7,8).

 If the length of the longer vector is not a multiple of the length of the shorter vector, R will still perform the operation but will issue a **warning message**.

In [ ]:
vec3 <- 1:5
vec4 <- 1:3

# The shorter vector (vec4) will be recycled. The length of vec3 (5) is not a multiple of vec4 (3).
# R will add (1,2,3,1,2) to (1,2,3,4,5), and issue a warning.
vec3 + vec4

### **3.3 Matrices**

Matrices are two-dimensional arrays that contain elements of the same type.

Assigning an "dim" attribute to a vector turns it into a matrix.

In [ ]:
# "mymat" is a vector
mymat <- 1:12
mymat
dim(mymat)

In [ ]:
# Assign a "dim" attribut to "mymat" turns it into a matrix.
dim(mymat) <- c(3,4)
mymat

In [ ]:
# Note the dual purposes of the dim() function.
# It displays the dimensions of a data object.
dim(mymat)
# It also provides a handle to assign values to dimensions.
dim(mymat) <- c(2,6)
mymat

Actually, matrices are merely vectors with a "dimension" attribute.

R matrices can be constructed by using the `matrix()` function as well.

In [ ]:
matrix(1:12,nrow=3,ncol=4)

The `matrix()` fucntion construct matrices column‐wise by default. You can use the "byrow=T" option to switch to row-wise.


In [ ]:
matrix(1:12,nrow=3,ncol=4,byrow=T)

Or by using the `cbind()` or `rbind()` functions.

In [ ]:
# Treat each argument as a column and bind them should-by-should into a matrix.
cbind(1:3,4:6,7:9,10:12)

In [ ]:
# Treat each argument as a row and bind them row-by-row into a matrix.
rbind(1:3,4:6,7:9,10:12)

One can use [\<index>,\<index>] to access individual elements.

In [ ]:
# What is the output?
mymat[3,4]

### **3.4 Lists**

Lists are an ordered collection of objects, which can be of **different types or classes**.

Lists can be constructed by using the `list()` function.


In [ ]:
# Mixing numeric, logical and character.
list(1,F,"a")

Members of a list do not have to be of atomic types, i.e. they can be vectors, matrices and even lists.

In [ ]:
my_list <- list(
  numeric_vector = c(1, 2, 3),
  char_vector = c("A", "B"),
  my_matrix = matrix(1:4, nrow = 2),
  nested_list = list(key = "value", number = 10)
)

# Print the structure of the list
str(my_list)

# Access elements
print(my_list$numeric_vector)
print(my_list$my_matrix)
print(my_list$nested_list$key)

Lists can be indexed using the `[[ ]]` operator.

In [ ]:
# The second element in mylist.
mylist[[2]]

The indices can be nested.

In [ ]:
# The second element of the fourth element (a list) in my list.
mylist[[4]][[2]]

Elements of R objects can have names.

Names can be specified when an object is created.

In [ ]:
list(inst="LSU",location="Baton Rouge",state="LA")

Or they can be specified later when the `names()` function.

In [ ]:
names(mylist) <- c("num","vec","mat","lst")
mylist

Names can be used to access elements in a data object using the `$` operator. When there are many elements, this could be more convenient than using the indices.

In [ ]:
# This is equivalent to mylist[[4]]
mylist$lst

 Indexing operations by names and indices can be nested and mixed.

In [ ]:
names(mylist$lst) <- c("c1","c2","c3")
mylist

In [ ]:
# The "c2" element of the "lst" element in the list "mylist".
mylist$lst$c2
# The same thing.
mylist$lst[[2]]
# Again, the same thing.
mylist[[4]][[2]]

### **3.5 Data Frames**

Data frames are used to store tabular data.
* They are a special type of **lists**, where each element is a R vector ("column" or "variable") and has to be of the same length.
* The elements (columns) can be of different classes.
* Data frames have special attributes such as row.names.
* Data frames can be created by reading data files, using functions such as `read.table()` or `read.csv()`.


Data frames can be created directly by using the `data.frame()` function.

In [ ]:
#@markdown **Data Frame**
# Build a small data frame from scratch
gene_df <- data.frame(
  gene    = genes,
  expr    = expr,
  is_high = high_expr,
  stringsAsFactors = FALSE
)

gene_df

# Basic inspection
nrow(gene_df)
ncol(gene_df)
str(gene_df)

Since data frames are lists, both `[[ ]]` and `$` operators work.

In [ ]:
gene_df[[2]]

In [ ]:
gene_df$expr

You could select rows and columns by leaving the other index blank:

In [ ]:
# The first two rows
gene_df[1:2,]

In [ ]:
# The "sex" column
gene_df[,"sex"]

We can subset a data frame like this:

In [ ]:
# Both columns for obs 1 and 3
gene_df[c(1,3),]

Or using a vector of logical values:

In [ ]:
# This gives us a logical vector.
gene_df$expr == "MYC"

In [ ]:
gene_df[gene_df$expr == "MYC",]

## **3.6 Querying Object Attributes**

There are a few functions in R that help us obtain information about an object.

We will work with the "mtcars" data frame in this section.

In [ ]:
# Get some help information on the data frame.
#?mtcars

# Print the data frame to screen.
#mtcars

# Length of the object
#length(mtcars)

# Number of rows in a data frame.
#nrow(mtcars)

# Dimension of an object
#dim(mtcars)

# Attributes of an object
#attributes(mtcars)

# Shows the internal strucutre of a R object
#str(mtcars)

In [ ]:
attach(mtcars)

## **3.7 Exercise 1**



1. Learn about the **airquality** data frame and answer the following quetions:
  * What is the source of the data?
  * How many rows and columns are there in the data frame?
  * What does each column represent?
2. Find the percentage of days when the high tempature measured at La Guardia Airport exceeded 70.
3. Find the number of days when the wind speed is between 10 and 20 miles per hour.

In [ ]:
#@markdown **Hint**
# Use a logical vector as indices to subset a data frame.

In [ ]:
#@markdown Solution { display-mode: "form" }
# question 1
?airquality

#attach(airquality)

# question 2
#nrow(airquality[Temp > 70,])/nrow(airquality)*100

# question 3
#nrow(airquality[Wind > 10 & Wind < 20,])

#detach(airquality)

---

# **4· Data Wrangling with Real Data**

Data wrangling means preparing and transforming data so it is easier to analyze.

In real data analysis, we often need to:

- select specific rows or columns
- filter data based on conditions
- check missing values and duplicates
- create new columns
- sort data
- summarize data

We'll use the built-in `mtcars` dataset (car performance specs) — small enough to understand
instantly, realistic enough to show real patterns.


In [ ]:
# Load and preview
data(mtcars)
head(mtcars)

### **4.1 Data Cleaning**

Data cleaning focuses on fixing problems in the data.

Examples:

- checking missing values
- removing duplicate rows
- renaming columns
- fixing incorrect values

**Missing Values**

Missing values (missing data) are a common problem in data science. It can have a significant effect on the conclusions that can be drawn from the data. Therefore, R has many functions and packages that deal with the missing value problem.

Individual missing values can be identified using the `is.na()` function, which returns `TRUE` for `NA` values and `FALSE` otherwise. To count missing values per column, `colSums()` can be used in conjunction with `is.na()`.

In [ ]:
is.na(airquality) # Check for NA values in the entire dataframe

# Count NA values per column
colSums(is.na(airquality))

 The `complete.cases` function will scan a dataframe and return a logical vector where the rows without missing values are TRUE and those with missing values FALSE.

In [ ]:
# In the airquality dataset we have used for Exercise 1, there are some missing values.
complete.cases(airquality)

The simplest way of dealing with missing values is to drop all rows with missing values (not necessarily the best way!).

In [ ]:
airquality[complete.cases(airquality),]

**Duplicate Values**

In [ ]:
duplicated(airquality)
airquality[!duplicated(airquality), ]

### **4.2 Data Manipulation**

Data manipulation focuses on selecting, changing, organizing, and summarizing data.

Examples:

- creating new columns
- subsetting
- sorting rows
- filtering rows
- selecting columns
- summarizing data

**Creating New Columns**

To add columns/variables to a dataframe, we can simply use the assignment operation.

In [ ]:
gene_df <- data.frame(
  gene    = genes,
  expr    = expr,
  is_high = high_expr,
  stringsAsFactors = FALSE
)

In [ ]:
gene_df$expr_category <- ifelse(gene_df$expr > 40, "High", "Low")
gene_df

In [ ]:
gene_df$very_high <- gene_df$expr > 80
gene_df

In [ ]:
gene_df$expr_norm <- gene_df$expr / max(gene_df$expr)
gene_df

**Subsetting**

Subsetting means selecting specific parts of your data like a vector, data frame, or list in R.

You can subset:

* By column name
* By position `[row, col]`
* By condition

In [ ]:
# 1. By column name (most readable)
gene_df$expr

# 2. By position [row, col]
gene_df[1, ]      # first row
gene_df[, 2]      # second column

# 3. By condition
gene_df[gene_df$expr > 40, ]

The `subset()` function provides a more convenient way to select rows and columns from data frames based on conditions, often leading to more readable code.

In [ ]:
subset(gene_df, subset = expr > 40, select = c(gene, expr))

**Sorting**

Sort and order elements: `sort()`, `rank()` and `order(`).

By default, the `sort()` function sorts the values in a vector into ascending order.

In [ ]:
# This is the original order.
mtcars$mpg
# This is the sorted order.
sort(mtcars$mpg)

The ```decreasing=T``` option will sort a vector into descending order.

In [ ]:
sort(mtcars$mpg, decreasing=T)

In contrast, the `order()` function returns the indices in order.


In [ ]:
#order(mtcars$mpg)
order(mtcars$mpg, decreasing=T)

Users can use the indices returned by `order()` to change the order of rows in a data frame.

In [ ]:
# The data frame in the original order.
mtcars

In [ ]:
# The reorder data fram according the values of the mpg variable.
mtcars[order(mtcars$mpg),]

The `rank()` function returns the ranks of the values in a vector.


In [ ]:
#rank(mtcars$mpg, ties.method = "min")
#rank(mtcars$mpg)

In [ ]:
# The "ties.method" option decides how equal values are handled.
#data.frame(mpg=mtcars$mpg,rank=rank(mtcars$mpg, ties.method = "min"))

We'll use **dplyr** — the tidyverse data wrangling package.
Its verbs (`filter`, `select`, `mutate`, `summarize`, `arrange`) map almost directly
to SQL.

In [ ]:
# --- dplyr verbs ---

# filter(): keep rows matching a condition
efficient <- filter(mtcars, mpg > 25)
efficient

# select(): keep only certain columns
select(mtcars, mpg, cyl, hp)

# mutate(): add or transform a column
mtcars <- mutate(mtcars, kpl = mpg * 0.425)  # miles/gallon → km/litre
head(mtcars[, c("mpg", "kpl")])

In [ ]:
# arrange(): sort rows
arrange(mtcars, desc(mpg))   # best fuel efficiency first

## **4.3 Exercise 2**

Using the **airquality** data, find the average wind speed for the 10 hottest days on record.



In [ ]:
#@markdown **Hint**
# Use the order() function
# to find the indices of the 10 hottest days.

In [ ]:
#@markdown **Solution** { display-mode: "form" }
mean(airquality[order(airquality$Temp,decreasing=T),"Wind"][1:10])

**The pipe `|>`**

The pipe chains operations left-to-right — reads like a recipe.

```r
data |>
  step1() |>
  step2() |>
  step3()
```

> Read them top-to-bottom: each step's output becomes the next step's input.

In [ ]:
# summarize() + group_by(): aggregate statistics by group
mtcars |>
  group_by(cyl) |>
  summarize(
    n         = n(),
    mean_mpg  = round(mean(mpg), 1),
    mean_hp   = round(mean(hp), 1),
    mean_wt   = round(mean(wt), 2)
  )

**Quick exercise**

> Using the code above as a template, find the average horsepower (`hp`) and weight (`wt`)
for cars with 4 and 6 cylinders only (exclude `cyl == 8`).

In [ ]:
#@markdown **Hint**
# chain filter() before group_by()

In [ ]:
#@markdown **Solution** { display-mode: "form" }
mtcars %>%
  filter(cyl != 8) %>%
  group_by(cyl) %>%
  summarize(
    n        = n(),
    mean_mpg = round(mean(mpg), 1),
    mean_hp  = round(mean(hp), 1),
    mean_wt  = round(mean(wt), 2)
  )